In [ ]:
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic

client = Anthropic(
#     api_key= load_dotenv().get("ANTHROPIC_API_KEY")
) 

model = "claude-haiku-4-5-20251001"

message = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": "Write a haiku about the beauty of nature."
        }
    ]
)

print(message.content[0].text)  

In [ ]:
def add_user_message(conversation, content):
    conversation.append({
        "role": "user",
        "content": content
    })


def add_assistant_message(conversation, content):
    conversation.append({
        "role": "assistant",
        "content": content.replace("**", "").replace("##", " ")
    })


def chat(conversation, system=None, stop_sequences=None):

    params = {
        "model": model,
        "max_tokens": 300,
        "messages": conversation
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)

    text = response.content[0].text

    add_assistant_message(conversation, text)

    return text

In [ ]:
def chat_stream(conversation, system=None, stop_sequences=None):

    params = {
        "model": model,
        "max_tokens": 300,
        "messages": conversation
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    full_response = ""

    with client.messages.stream(**params) as stream:

        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_response += text

    print()

    add_assistant_message(conversation, full_response)

    return full_response

In [ ]:
def ask_ai(question, system=None, stop_sequences=None,stream = False):

    add_user_message(conversation, question)

    if stream:
        return chat_stream(
            conversation,
            system=system,
            stop_sequences=stop_sequences
        )
    else:
        return chat(
            conversation,
            system=system,
            stop_sequences=stop_sequences
    )

In [ ]:


add_user_message(conversation, "tell me about death note anime?")

response = chat(conversation)
print(response)

add_assistant_message(conversation, response)

add_user_message(conversation, "who is the main character?")
response = chat(conversation)
print(response)

In [ ]:
conversation = []

input_question = input("Enter your first question: ")
response = ask_ai(input_question)
print("First response -----------")
print(response)

input_question = input("Enter your second question: ")
response = ask_ai(input_question)
print("Second response -----------")
print(response)


In [ ]:
conversation = []

system_maths_tuter = "You are a helpful and precise assistant for solving math problems. You will be provided with a math problem, and you will respond with the step-by-step solution to the problem. You should only provide the solution and not any additional information or explanations. start responce by 'this is math mentoer' and end responce by 'end of solution'."

# input_question = input("Enter your math problem: ")
input_question = "**Solve the equation 2x + 3 = 7 for x.**"
print("Math problem without system message -----------")
response = ask_ai(input_question)
print(response)
print("Math problem with system message -----------")
response = ask_ai(input_question, system=system_maths_tuter)
print(response)

In [ ]:

def add_user_message(conversation, content):
    conversation.append({
        "role": "user",
        "content": content
    })
    
def add_assistant_message(conversation, content):
    conversation.append({
        "role": "assistant",
        "content": content.replace("**", "").replace("##", " ")
    })
    
def chat(conversation ,system = None):
    params = {
        "model": model,
        "max_tokens": 300,
        "messages": conversation,
        "stream": True
     }
    if system:
        params["system"] = system
        
    response = client.messages.create(
        **params
    )
    # add_assistant_message(conversation, response.content[0].text)  
    return response

# def ask_ai(question , system = None):
#     add_user_message(conversation, question)
#     response = chat(conversation, system if system else None)
#     return response
conversation = []

add_user_message(conversation, "**give me one line summary of one punch man anime.**")
stream = chat(conversation)

for message in stream:
    print(message, end="")

In [ ]:
conversation = []

add_user_message(conversation, "tell me summary  about one punch man anime?")

with client.messages.stream(
    model=model,
    max_tokens=300,
    messages=conversation
) as stream:
    for message in stream.text_stream:
        # print(message.replace("**", "").replace("##", " "), end="")
        pass
    print("Final response: ", stream.get_final_text())
    conversation.append({
        "role": "assistant",
        "content": stream.get_final_text()
    })

In [ ]:
conversation

In [ ]:
conversation = []



add_user_message(conversation, "give me very short event bridge rule json ")
add_assistant_message(conversation, "```json" )
chat_response = chat(conversation,stop_sequences=["```"])
print(chat_response)

